# 08 — REST end-to-end: ingest, aggregate, retrieve (SDK only)

This notebook uses **only** [`nostradamus-ioto-sdk`](https://github.com/robotics-4-all/nostradamus-ioto-sdk) for every REST call.
No `requests`, no raw `httpx` — the SDK is the whole client surface.

Flow:

1. Provision: organization guard → project → collection → scoped API keys
2. Ingest: batch `POST /send_data` with a **write** key
3. Retrieve: `GET /get_data` with filters, attribute projection, ordering, pagination (with a **read** key)
4. Aggregate: `GET /statistics` — `avg`/`max`/`count`/`p95` over time buckets, `group_by`, `distinct`
5. Delete by key and prove the row vanished
6. Clean up everything this notebook created

Prerequisites:

```bash
pip install -r examples/notebooks/requirements.txt
cp examples/notebooks/env.example .env   # set CENOTOO_USERNAME / CENOTOO_PASSWORD
```

This notebook targets the **currently exposed TLS interface**:

| Service | URL |
|---|---|
| REST API | `https://api.cenotoo.local` |
| Trust | `cenotoo-local-ca.crt` (self-signed local CA) |

If your deployment uses a public ACME certificate, leave `CENOTOO_CA_BUNDLE` unset.
For the local self-signed deployment, keep the default CA path or point
`CENOTOO_CA_BUNDLE` at the CA file. The notebook exports it as `SSL_CERT_FILE`
so the SDK's token fetch and API calls trust the endpoint.

If the environment variables are missing the notebook stops with a `SKIP` message.

In [ ]:
%pip install -q "nostradamus-ioto-sdk @ git+https://github.com/robotics-4-all/nostradamus-ioto-sdk.git@4f8e7fc7c28c859eefed9f7382d1b148a5c6eeb8" python-dotenv

## 1. Imports, configuration and helpers

In [ ]:
import json
import os
import random
import time
from datetime import datetime, timedelta, timezone

from dotenv import load_dotenv
from nostradamus_ioto_sdk import NostradamusClient
from nostradamus_ioto_sdk.models import KeyType

load_dotenv()

BASE_URL = os.getenv("CENOTOO_BASE_URL", "https://api.cenotoo.local")
USERNAME = os.getenv("CENOTOO_USERNAME", "")
PASSWORD = os.getenv("CENOTOO_PASSWORD", "")
assert USERNAME and PASSWORD, "SKIP: set CENOTOO_USERNAME and CENOTOO_PASSWORD (see env.example)"


def resolve_ca_bundle():
    configured = os.getenv("CENOTOO_CA_BUNDLE", "")
    for candidate in (configured, "cenotoo-local-ca.crt", "../../cenotoo-local-ca.crt"):
        if candidate and os.path.isfile(os.path.abspath(os.path.expanduser(candidate))):
            return os.path.abspath(os.path.expanduser(candidate))
    return None


CA_BUNDLE = None
if ".local" in BASE_URL or os.getenv("CENOTOO_CA_BUNDLE"):
    CA_BUNDLE = resolve_ca_bundle()
    if CA_BUNDLE:
        os.environ["SSL_CERT_FILE"] = CA_BUNDLE

TLS_KWARGS = {"verify": CA_BUNDLE} if CA_BUNDLE else {}


def unique_name(prefix):
    return f"{prefix}_{time.strftime('%Y%m%d%H%M%S')}_{random.randint(1000, 9999)}"


def wait_for(fn, predicate, timeout=120, interval=3, description="condition"):
    """Poll an SDK call until Cassandra's asynchronous write path catches up."""
    deadline = time.monotonic() + timeout
    last = None
    while time.monotonic() < deadline:
        last = fn()
        if predicate(last):
            return last
        time.sleep(interval)
    raise TimeoutError(f"timed out waiting for {description}; last={last!r}")


def show(rows, title="", limit=20):
    """Tiny dependency-free table printer for notebook output."""
    if title:
        print(f"\n{title}")
    if isinstance(rows, dict):
        rows = [rows]
    rows = list(rows or [])[:limit]
    if not rows:
        print("  (no rows)")
        return
    columns = list(rows[0].keys())
    print("  " + " | ".join(columns))
    for row in rows:
        print("  " + " | ".join(str(row.get(column)) for column in columns))

## 2. Provision project, collection and scoped keys

The collection schema is a **template document**: Python types map to Cassandra types
(`float` → double, `int` → int, `str` → text). `key`, `timestamp`, `day` and `id`
are managed by the platform.

The SDK uses OAuth2 with `username`/`password` for administrative calls.

In [ ]:
admin = NostradamusClient(base_url=BASE_URL, username=USERNAME, password=PASSWORD, **TLS_KWARGS)
organization = admin.organizations.get()

project = admin.projects.create(
    name=unique_name("rest_demo"),
    description="REST SDK end-to-end example",
    tags=["notebook", "rest", "sdk"],
)

COLLECTION_SCHEMA = {
    "temperature": 0.0,     # float -> Cassandra double
    "humidity": 0,          # int   -> Cassandra int
    "soil_moisture": 0.0,   # float -> Cassandra double
    "room": "north",        # str   -> Cassandra text
}

collection = admin.collections.create(
    project_id=project.project_id,
    name=unique_name("greenhouse"),
    description="Greenhouse sensor readings",
    collection_schema=COLLECTION_SCHEMA,
    tags=["notebook", "rest", "sdk"],
)

write_key = admin.project_keys.create(project.project_id, KeyType.WRITE)
read_key = admin.project_keys.create(project.project_id, KeyType.READ)

PROJECT_ID = project.project_id
COLLECTION_ID = collection.collection_id

print("organization :", organization.organization_name)
print("project      :", project.project_name, PROJECT_ID)
print("collection   :", collection.collection_name, COLLECTION_ID)
print("schema       :", collection.collection_schema)

## 3. Switch to scoped API-key clients

Devices and services should not carry user credentials. The SDK accepts an `api_key`
in place of `username`/`password`, so the rest of this notebook runs with the
least-privilege keys created above.

In [ ]:
write_client = NostradamusClient(base_url=BASE_URL, api_key=write_key.api_key, **TLS_KWARGS)
read_client = NostradamusClient(base_url=BASE_URL, api_key=read_key.api_key, **TLS_KWARGS)
print("scoped clients ready (write + read)")

## 4. Data ingestion

Send a batch of six readings per room as a single `POST send_data` call.
Each record carries a stable `key` (the sensor/device id) and an explicit ISO-8601
`timestamp`; the write key has just enough scope for ingestion.

In [ ]:
ROOMS = {
    "north": {"temperature": 21.0, "humidity": 55},
    "south": {"temperature": 25.0, "humidity": 42},
}
base_time = datetime.now(timezone.utc).replace(microsecond=0) - timedelta(hours=3)

records = []
for room, start in ROOMS.items():
    for step in range(6):
        records.append(
            {
                "key": f"{room}-sensor-01",
                "timestamp": (base_time + timedelta(minutes=30 * step))
                .isoformat()
                .replace("+00:00", "Z"),
                "temperature": round(start["temperature"] + 0.8 * step, 2),
                "humidity": start["humidity"] + step,
                "soil_moisture": round(
                    35.0 + 1.2 * step + (0.0 if room == "north" else 4.0), 2
                ),
                "room": room,
            }
        )

write_client.data.send(PROJECT_ID, COLLECTION_ID, records)
print(f"sent {len(records)} readings via POST send_data")
print("sample:", json.dumps(records[0], indent=2))

Ingestion is asynchronous: REST writes land in Kafka first, then the consumer
writes them to Cassandra. Poll `get_data` until every record is readable.

In [ ]:
def visible_count():
    return read_client.data.get(PROJECT_ID, COLLECTION_ID, limit=100).total_count


total = wait_for(
    visible_count,
    lambda count: count >= len(records),
    description="all readings to become readable",
)
print("readable via GET get_data:", total, "records")

## 5. Data retrieval

`data.get()` maps directly to `GET /get_data`:

| SDK argument | API parameter | Meaning |
|---|---|---|
| `attributes=[...]` | `attributes` | project only the fields you need |
| `filters=[...]` | `filters` (JSON) | `eq`, `ne`, `lt`, `lte`, `gt`, `gte`, `in`, `contains` (AND-ed) |
| `order_by=json.dumps({"field": ..., "order": ...})` | `order_by` | `asc` / `desc` |
| `limit=N` | `limit` | page size |

The next two calls show the two most common shapes: "latest N readings" and
"filtered + ordered readings".

In [ ]:
latest = read_client.data.get(
    PROJECT_ID,
    COLLECTION_ID,
    attributes=["timestamp", "room", "temperature", "humidity", "soil_moisture"],
    order_by=json.dumps({"field": "timestamp", "order": "desc"}),
    limit=5,
)
show(latest.data, f"Latest 5 readings (total_count={latest.total_count})")

In [ ]:
warm_north = read_client.data.get(
    PROJECT_ID,
    COLLECTION_ID,
    attributes=["timestamp", "room", "temperature"],
    filters=[
        {"property_name": "room", "operator": "eq", "property_value": "north"},
        {"property_name": "temperature", "operator": "gte", "property_value": 24.0},
    ],
    order_by=json.dumps({"field": "temperature", "order": "desc"}),
    limit=10,
)
show(warm_north.data, "North room, temperature >= 24.0 (hottest first)")

## 6. Data aggregation

`data.statistics()` maps to `GET /statistics`. Supported operations:
`avg`, `max`, `min`, `sum`, `count`, `distinct`, and percentiles `p50`, `p90`, `p95`, `p99`.

- `interval` uses `every_N_<minutes|hours|days|weeks|months>` (for example `every_1_hours`).
- `group_by` defaults to `key`; here we group by `room`.
- Every bucket contains `interval_start` plus `<stat>_<attribute>` (for example `avg_temperature`).
- `stat="distinct"` returns per-key statistics instead of buckets.

In [ ]:
def aggregate(**kwargs):
    return read_client.data.statistics(PROJECT_ID, COLLECTION_ID, **kwargs)


hourly_avg = aggregate(
    stat="avg", attribute="temperature", interval="every_1_hours", group_by="room"
)
show(hourly_avg, "Average temperature per room, 1-hour buckets")

hourly_max = aggregate(
    stat="max", attribute="humidity", interval="every_1_hours", group_by="room"
)
show(hourly_max, "Max humidity per room, 1-hour buckets")

counts = aggregate(
    stat="count", attribute="temperature", interval="every_1_hours", group_by="room"
)
show(counts, "Reading count per room, 1-hour buckets")

p95 = aggregate(
    stat="p95", attribute="temperature", interval="every_1_hours", group_by="room"
)
show(p95, "p95 temperature per room, 1-hour buckets")

distinct = aggregate(stat="distinct", attribute="room")
show(distinct.get("key_statistics", []), "Distinct rooms (per sensor key)")

> **SDK scope note:** the current SDK exposes `stat`/`attribute`/`group_by`/`interval`/`limit`
> for `statistics`. Server-side `start_time` / `end_time` / `filters` / `order` are not
> modeled by the SDK yet; use `interval` + `group_by` for the examples above.

## 7. Delete data by key

`data.delete()` accepts a `key`, a timestamp range, or both. Here we delete one
sensor's records and poll until Cassandra no longer returns them.

In [ ]:
admin.data.delete(PROJECT_ID, COLLECTION_ID, key="north-sensor-01")


def north_rows():
    response = read_client.data.get(
        PROJECT_ID,
        COLLECTION_ID,
        attributes=["timestamp", "room", "temperature"],
        filters=[
            {
                "property_name": "key",
                "operator": "eq",
                "property_value": "north-sensor-01",
            }
        ],
        limit=10,
    )
    return response.total_count


wait_for(north_rows, lambda count: count == 0, description="deleted key to vanish")
remaining = read_client.data.get(PROJECT_ID, COLLECTION_ID, limit=100).total_count
print("delete by key OK — remaining records:", remaining)

## 8. Clean up

In [ ]:
import contextlib

for client in (write_client, read_client):
    with contextlib.suppress(Exception):
        client.close()

with contextlib.suppress(Exception):
    admin.project_keys.delete(PROJECT_ID, read_key.api_key)
with contextlib.suppress(Exception):
    admin.project_keys.delete(PROJECT_ID, write_key.api_key)
with contextlib.suppress(Exception):
    admin.collections.delete(PROJECT_ID, COLLECTION_ID)
with contextlib.suppress(Exception):
    admin.projects.delete(PROJECT_ID)
with contextlib.suppress(Exception):
    admin.close()

print("cleanup done")